# Data Exploration

In [30]:
!pip install -q python-terrier ir_datasets pandas nltk

In [31]:
import pyterrier as pt
import pandas as pd
import nltk
from nltk.corpus import stopwords
import re

In [32]:
dataset_name = "irds:cord19/trec-covid"
dataset = pt.get_dataset(dataset_name)

print(f"Loaded dataset: {dataset_name}")
irds_dataset = dataset.irds_ref()

Loaded dataset: irds:cord19/trec-covid


## Query Exploration
### Look at Verbosity Levels

In [33]:
topics_data = []
for query in irds_dataset.queries_iter():
    topics_data.append({
        # 'qid': query.query_id,
        'title': query.title,
        'description': query.description,
        'narrative': query.narrative
    })

df_topics = pd.DataFrame(topics_data)
print(f"Total number of queries: {len(df_topics)}")
df_topics.head(3)

Total number of queries: 50


,title,description,narrative
0,coronavirus origin,what is the origin of COVID-19,seeking range of information about the SARS-Co...
1,coronavirus response to weather changes,how does the coronavirus respond to changes in...,seeking range of information about the SARS-Co...
2,coronavirus immunity,will SARS-CoV2 infected people develop immunit...,seeking studies of immunity developed due to i...


### Average Length of the Query at each Verbosity Level Before and After Stopword Removal

In [34]:
nltk.download('stopwords', quiet=True)
stop_words = set(stopwords.words('english'))

def count_words(text):
    if pd.isna(text): return 0
    return len(str(text).split())

def count_words_without_stopwords(text):
    if pd.isna(text): return 0
    words = str(text).lower().split()
    filtered_words = [w for w in words if w not in stop_words]
    return len(filtered_words)

In [35]:
verbosity_levels = ['title', 'description', 'narrative']

for level in verbosity_levels:
    df_topics[f'{level}_len'] = df_topics[level].apply(count_words)
    df_topics[f'{level}_len_no_stops'] = df_topics[level].apply(count_words_without_stopwords)


print("--- Average Query Length (INCLUDING Stopwords) ---")
for level in verbosity_levels:
    avg_len = df_topics[f'{level}_len'].mean()
    print(f"{level.capitalize()}: {avg_len:.2f} words")

print("\n--- Average Query Length (EXCLUDING Stopwords) ---")
for level in verbosity_levels:
    avg_len_no_stops = df_topics[f'{level}_len_no_stops'].mean()
    print(f"{level.capitalize()}: {avg_len_no_stops:.2f} words")

--- Average Query Length (INCLUDING Stopwords) ---
Title: 3.24 words
Description: 10.60 words
Narrative: 23.52 words

--- Average Query Length (EXCLUDING Stopwords) ---
Title: 2.74 words
Description: 5.56 words
Narrative: 14.74 words


## Relevance Judgments Exploration

In [36]:
qrels = dataset.get_qrels()

print(f"Total number of relevance judgments: {len(qrels)}")

print("\nRelevance Score Distribution:")
qrels['label'].value_counts()

Total number of relevance judgments: 69318

Relevance Score Distribution:


label
 0    42652
 2    15609
 1    11055
-1        2
Name: count, dtype: int64

## Document Exploration

In [37]:
doc_iter = irds_dataset.docs_iter()

sample_docs = []
for i, doc in enumerate(doc_iter):
    if i >= 5:
        break
    sample_docs.append({
        'docno': doc.doc_id,
        'title': doc.title,
        'abstract': doc.abstract[:300]
    })

df_docs = pd.DataFrame(sample_docs)
df_docs

,docno,title,abstract
0,ug7v899j,Clinical features of culture-proven Mycoplasma...,OBJECTIVE: This retrospective chart review des...
1,02tnwd4m,Nitric oxide: a pro-inflammatory mediator in l...,Inflammatory diseases of the respiratory tract...
2,ejv2xln0,Surfactant protein-D and pulmonary host defense,Surfactant protein-D (SP-D) participates in th...
3,2b73a28n,Role of endothelin-1 in lung disease,Endothelin-1 (ET-1) is a 21 amino acid peptide...
4,9785vg6d,Gene expression in epithelial cells in respons...,Respiratory syncytial virus (RSV) and pneumoni...


## Indexing

In [38]:
import os

if not os.path.exists("./index/data.properties"):
    indexer = pt.IterDictIndexer("./index")
    
    index_ref = indexer.index(
        ({
            "docno": d.doc_id,
            "text": (d.title or "") + " " + (d.abstract or "")
        }
        for d in irds_dataset.docs_iter())
    )
else:
    index_ref = "./index"

index = pt.IndexFactory.of(index_ref)

20:35:57.078 [ForkJoinPool-1-worker-3] WARN org.terrier.structures.indexing.Indexer -- Adding an empty document to the index (8is9x9sc) - further warnings are suppressed
20:36:26.360 [ForkJoinPool-1-worker-3] ERROR org.terrier.structures.indexing.Indexer -- Could not finish MetaIndexBuilder: 
java.io.IOException: Key 8lqzfj2e is not unique: 37597,11755
For MetaIndex, to suppress, set metaindex.compressed.reverse.allow.duplicates=true
	at org.terrier.structures.collections.FSOrderedMapFile$MultiFSOMapWriter.mergeTwo(FSOrderedMapFile.java:1374)
	at org.terrier.structures.collections.FSOrderedMapFile$MultiFSOMapWriter.close(FSOrderedMapFile.java:1308)
	at org.terrier.structures.indexing.BaseMetaIndexBuilder.close(BaseMetaIndexBuilder.java:321)
	at org.terrier.structures.indexing.classical.BasicIndexer.indexDocuments(BasicIndexer.java:270)
	at org.terrier.structures.indexing.classical.BasicIndexer.createDirectIndex(BasicIndexer.java:388)
	at org.terrier.structures.indexing.Indexer.index(In

In [39]:
#loading the index
index = pt.IndexFactory.of("./cord19_index/data.properties")
#or
#index = pt.IndexFactory.of(index_ref)

## Preprocessing

In [42]:
#performing preprocessing on the queries
import re
import nltk
from nltk.stem import PorterStemmer
from spellchecker import SpellChecker
from nltk.corpus import stopwords
from nltk import pos_tag

nltk.download('stopwords')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('punkt')

stemmer = PorterStemmer()
spell = SpellChecker()
stop_words = set(stopwords.words('english'))

def preprocess_text(text, level):
    if pd.isna(text):
        return ""
    
    # lowercasing
    text = text.lower()
    # remove punctuation
    text = re.sub(r'[^\w\s]', '', text)

    # level 0: just lowercase + remove punctuation
    if level == 'title':
        return text

    words = text.split()

    # level 1: POS filtering + stemming + stopword removal
    if level == 'description':
        pos_tags = pos_tag(words)
        allowed_pos_tags = {'NN', 'NNS', 'VBN', 'JJ', 'RB', 'NNP', 'NNPS'}
        filtered_pos_words = [token for token, pos in pos_tags if pos in allowed_pos_tags]
        stemmed_words = [stemmer.stem(token) for token in filtered_pos_words]
        filtered_words = [token for token in stemmed_words if token not in stop_words]

    # level 2: spell correction + POS filtering + stemming + stopword removal
    if level == 'narrative':
        corrected_words = [spell.correction(token) or token for token in words]
        pos_tags = pos_tag(corrected_words)
        allowed_pos_tags = {'NN', 'NNS', 'VBN', 'JJ', 'RB', 'NNP', 'NNPS'}
        filtered_pos_words = [token for token, pos in pos_tags if pos in allowed_pos_tags]
        stemmed_words = [stemmer.stem(token) for token in filtered_pos_words]
        filtered_words = [token for token in stemmed_words if token not in stop_words]

    return ' '.join(filtered_words)

print(verbosity_levels)

for level in verbosity_levels:
    print(f"\nPreprocessing with verbosity level: {level}")
    df_topics[f'preprocessed_title_{level}'] = df_topics['title'].apply(lambda x: preprocess_text(x, level))
    df_topics[f'preprocessed_description_{level}'] = df_topics['description'].apply(lambda x: preprocess_text(x, level))
    df_topics[f'preprocessed_narrative_{level}'] = df_topics['narrative'].apply(lambda x: preprocess_text(x, level))

print(df_topics.head(3))

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/nehiraltinkaya/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /Users/nehiraltinkaya/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package punkt to
[nltk_data]     /Users/nehiraltinkaya/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


['title', 'description', 'narrative']

Preprocessing with verbosity level: title

Preprocessing with verbosity level: description

Preprocessing with verbosity level: narrative
                                     title  \
0                       coronavirus origin   
1  coronavirus response to weather changes   
2                     coronavirus immunity   

                                         description  \
0                     what is the origin of COVID-19   
1  how does the coronavirus respond to changes in...   
2  will SARS-CoV2 infected people develop immunit...   

                                           narrative  title_len  \
0  seeking range of information about the SARS-Co...          2   
1  seeking range of information about the SARS-Co...          5   
2  seeking studies of immunity developed due to i...          2   

   title_len_no_stops  description_len  description_len_no_stops  \
0                   2                6                         2   
1       

In [45]:
#checking the effect of preprocessing on a sample query

sample = df_topics['title'][0]
print("Original:  ", sample)
print("Level 0:   ", preprocess_text(sample, 'title'))  # just lowercase + remove punctuation
print("Level 1:   ", preprocess_text(sample, 'description'))
print("Level 2:   ", preprocess_text(sample, 'narrative'))
print(preprocess_text("coronavyrus originn", 'description'))  # no spell correction
print(preprocess_text("coronavyrus originn", 'narrative'))  # spell correction applied first

Original:   coronavirus origin
Level 0:    coronavirus origin
Level 1:    coronaviru origin
Level 2:    coronaviru origin
coronavyru originn
coronavyru origin
